In [1]:
!pip install transformers accelerate torch torchvision

In [2]:
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from PIL import Image
import torch

class Blip2QA:
    def __init__(self, model_name="Salesforce/blip2-flan-t5-xl", dtype=torch.float16):
        self.processor = Blip2Processor.from_pretrained(model_name)
        self.model = Blip2ForConditionalGeneration.from_pretrained(
            model_name,
            device_map="auto",
            torch_dtype=dtype
        )

    def ask_question(self, image_path, question, caption=None, max_new_tokens=30):
        image = Image.open(image_path).convert("RGB")
        prompt = f"Question: {question}"
        inputs = self.processor(images=image, text=prompt, return_tensors="pt").to(self.model.device, torch.float16)
        
        with torch.no_grad():
            generated_ids = self.model.generate(**inputs, max_new_tokens=max_new_tokens)
            answer = self.processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
        
        return answer

qa_model = Blip2QA()
image_path = "/media/gp921526/Thanet/Work/Multi_Agent_Cartoon/dataset/simpsons/val_images/S33/S33E13-04701.jpg"
question = "what is in the background?"   
answer = qa_model.ask_question(image_path, question)
print("Answer:", answer)

/home/gp921526/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-07-12 18:03:54.174769: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-12 18:03:54.182869: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752339834.196552   42171 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752339834.199865   42171 cuda_blas.cc:1407] Unab

Answer: a board game
